# Feature Engineering — Comprehensive Practical Notebook

### Dataset: KC House Data

This notebook is designed as a **machine-learning training reference**. It demonstrates how raw columns can be transformed into useful machine-learning features using simple, readable Python.

We cover:

- Numerical transformations
- Ratios and domain features
- Date/time feature extraction
- Age and renovation features
- Categorical encoding
- Frequency/count encoding
- Binning
- Log and power transformations
- Interaction and polynomial features
- Missing-value indicators
- Outlier-aware transformations
- Latitude/longitude and geographic features
- Distance and spatial features
- Aggregation features
- Scaling/normalization
- Train/test leakage precautions
- Feature selection basics
- A reusable feature-engineering pipeline
- A final before/after feature summary

> **Important:** Feature engineering is not simply "creating columns". It includes **creating, transforming, extracting, combining, encoding, and selecting features** so that the data becomes more useful for a machine-learning model.


In [5]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    OneHotEncoder,
    PolynomialFeatures
)

from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 100)

# Load the dataset
df = pd.read_csv("KC_House_Data.csv")

df


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,20141013T000000,7700000,6,8.00,12050,27600,2.5,0,3,4,13,8570,3480,1910,1987,98102,47.6298,-122.323,3940,8800
1,20140611T000000,7062500,5,4.50,10040,37325,2.0,1,2,3,11,7680,2360,1940,2001,98004,47.6500,-122.214,3930,25449
2,20140919T000000,6885000,6,7.75,9890,31374,2.0,0,4,3,13,8860,1030,2001,0,98039,47.6305,-122.240,4540,42730
3,20140804T000000,5570000,5,5.75,9200,35069,2.0,0,0,3,13,6200,3000,2001,0,98039,47.6289,-122.233,3560,24345
4,20150413T000000,5350000,5,5.00,8000,23985,2.0,0,4,3,12,6720,1280,2009,0,98004,47.6232,-122.220,4600,21750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21608,20141105T000000,82000,3,1.00,860,10426,1.0,0,0,3,6,860,0,1954,0,98146,47.4987,-122.341,1140,11250
21609,20150324T000000,81000,2,1.00,730,9975,1.0,0,0,1,5,730,0,1943,0,98168,47.4808,-122.315,860,9000
21610,20140523T000000,80000,1,0.75,430,5050,1.0,0,0,2,4,430,0,1912,0,98014,47.6499,-121.909,1200,7500
21611,20140506T000000,78000,2,1.00,780,16344,1.0,0,0,1,5,780,0,1942,0,98168,47.4739,-122.280,1700,10387


## 2. Understand the raw variables

Before engineering features, identify:

- Numerical variables
- Categorical variables
- Date/time variables
- Geographic variables
- Potential target variable
- ID-like variables
- Variables that may cause leakage

For this dataset, `price` is the natural **target variable** for a house-price prediction exercise. We should normally not use the target to create input features.


In [6]:
df.shape

(21613, 20)

In [7]:
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,20141013T000000,7700000,6,8.00,12050,27600,2.5,0,3,4,13,8570,3480,1910,1987,98102,47.6298,-122.323,3940,8800
1,20140611T000000,7062500,5,4.50,10040,37325,2.0,1,2,3,11,7680,2360,1940,2001,98004,47.6500,-122.214,3930,25449
2,20140919T000000,6885000,6,7.75,9890,31374,2.0,0,4,3,13,8860,1030,2001,0,98039,47.6305,-122.240,4540,42730
3,20140804T000000,5570000,5,5.75,9200,35069,2.0,0,0,3,13,6200,3000,2001,0,98039,47.6289,-122.233,3560,24345
4,20150413T000000,5350000,5,5.00,8000,23985,2.0,0,4,3,12,6720,1280,2009,0,98004,47.6232,-122.220,4600,21750


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           21613 non-null  object 
 1   price          21613 non-null  int64  
 2   bedrooms       21613 non-null  int64  
 3   bathrooms      21613 non-null  float64
 4   sqft_living    21613 non-null  int64  
 5   sqft_lot       21613 non-null  int64  
 6   floors         21613 non-null  float64
 7   waterfront     21613 non-null  int64  
 8   view           21613 non-null  int64  
 9   condition      21613 non-null  int64  
 10  grade          21613 non-null  int64  
 11  sqft_above     21613 non-null  int64  
 12  sqft_basement  21613 non-null  int64  
 13  yr_built       21613 non-null  int64  
 14  yr_renovated   21613 non-null  int64  
 15  zipcode        21613 non-null  int64  
 16  lat            21613 non-null  float64
 17  long           21613 non-null  float64
 18  sqft_l

In [9]:
df.describe()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
count,2.161300e+04,21613.000000,21613.000000,21613.000000,2.161300e+04,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000,21613.000000
mean,5.400881e+05,3.370842,2.114757,2079.899736,1.510697e+04,1.494309,0.007542,0.234303,3.409430,7.656873,1788.390691,291.509045,1971.005136,84.402258,98077.939805,47.560053,-122.213896,1986.552492,12768.455652
std,3.671272e+05,0.930062,0.770163,918.440897,4.142051e+04,0.539989,0.086517,0.766318,0.650743,1.175459,828.090978,442.575043,29.373411,401.679240,53.505026,0.138564,0.140828,685.391304,27304.179631
min,7.500000e+04,0.000000,0.000000,290.000000,5.200000e+02,1.000000,0.000000,0.000000,1.000000,1.000000,290.000000,0.000000,1900.000000,0.000000,98001.000000,47.155900,-122.519000,399.000000,651.000000
25%,3.219500e+05,3.000000,1.750000,1427.000000,5.040000e+03,1.000000,0.000000,0.000000,3.000000,7.000000,1190.000000,0.000000,1951.000000,0.000000,98033.000000,47.471000,-122.328000,1490.000000,5100.000000
50%,4.500000e+05,3.000000,2.250000,1910.000000,7.618000e+03,1.500000,0.000000,0.000000,3.000000,7.000000,1560.000000,0.000000,1975.000000,0.000000,98065.000000,47.571800,-122.230000,1840.000000,7620.000000
75%,6.450000e+05,4.000000,2.500000,2550.000000,1.068800e+04,2.000000,0.000000,0.000000,4.000000,8.000000,2210.000000,560.000000,1997.000000,0.000000,98118.000000,47.678000,-122.125000,2360.000000,10083.000000
max,7.700000e+06,33.000000,8.000000,13540.000000,1.651359e+06,3.500000,1.000000,4.000000,5.000000,13.000000,9410.000000,4820.000000,2015.000000,2015.000000,98199.000000,47.777600,-121.315000,6210.000000,871200.000000


In [10]:
df.describe(include='object')

,date
count,21613
unique,372
top,20140623T000000
freq,142


In [11]:
df.dtypes

date              object
price              int64
bedrooms           int64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
grade              int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
zipcode            int64
lat              float64
long             float64
sqft_living15      int64
sqft_lot15         int64
dtype: object

In [12]:
df.isnull().sum().sort_values(ascending=False)

date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
grade            0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
zipcode          0
lat              0
long             0
sqft_living15    0
sqft_lot15       0
dtype: int64

# Part A — Date Feature Engineering

A date is often much more useful after extracting meaningful components.

From one date column we can create:

- Year
- Month
- Day
- Day of week
- Weekend indicator
- Quarter
- Month name
- Day of year
- Week number
- Days since a reference date
- Time elapsed between dates

For ML, numeric representations are often easier to use than raw date strings.


In [13]:
# 3. Convert the date column to datetime

df["date"] = pd.to_datetime(df["date"], errors="coerce")

display(df[["date"]].head())
print("Missing dates:", df["date"].isna().sum())


,date
0,2014-10-13
1,2014-06-11
2,2014-09-19
3,2014-08-04
4,2015-04-13


Missing dates: 0


In [14]:
# 4. Extract date components

df["sale_year"] = df["date"].dt.year
df["sale_month"] = df["date"].dt.month
df["sale_day"] = df["date"].dt.day
df["sale_quarter"] = df["date"].dt.quarter
df["sale_dayofweek"] = df["date"].dt.dayofweek
df["sale_dayofyear"] = df["date"].dt.dayofyear
df["sale_week"] = df["date"].dt.isocalendar().week.astype("int64")

# Monday=0 ... Sunday=6
df["is_weekend"] = (df["sale_dayofweek"] >= 5).astype(int)

# Month/weekday names can be useful for analysis,
# but categorical encoding may be required before many ML models.
df["sale_month_name"] = df["date"].dt.month_name()
df["sale_weekday_name"] = df["date"].dt.day_name()

display(df[[
    "date", "sale_year", "sale_month", "sale_day",
    "sale_quarter", "sale_dayofweek", "is_weekend",
    "sale_month_name", "sale_weekday_name"
]].head())


,date,sale_year,sale_month,sale_day,sale_quarter,sale_dayofweek,is_weekend,sale_month_name,sale_weekday_name
0,2014-10-13,2014,10,13,4,0,0,October,Monday
1,2014-06-11,2014,6,11,2,2,0,June,Wednesday
2,2014-09-19,2014,9,19,3,4,0,September,Friday
3,2014-08-04,2014,8,4,3,0,0,August,Monday
4,2015-04-13,2015,4,13,2,0,0,April,Monday


# Part B — Age and Time-Difference Features

A raw year such as `yr_built = 1985` is less intuitive than:

> **House age at the time of sale = Sale year − Year built**

This is a classic example of **domain-driven feature engineering**.


In [15]:
# 6. House age and renovation features

df["house_age"] = df["sale_year"] - df["yr_built"]

# yr_renovated is 0 when the house was not renovated in this dataset.
df["was_renovated"] = (df["yr_renovated"] > 0).astype(int)

display(df[[
    "yr_built", "yr_renovated", "sale_year",
    "house_age", "was_renovated"
]].head())


,yr_built,yr_renovated,sale_year,house_age,was_renovated
0,1910,1987,2014,104,1
1,1940,2001,2014,74,1
2,2001,0,2014,13,0
3,2001,0,2014,13,0
4,2009,0,2015,6,0


# Part C — Numerical Feature Engineering

Common numerical transformations include:

1. Ratios
2. Differences
3. Sums
4. Products
5. Log transformation
6. Square-root transformation
7. Power transformation
8. Binning
9. Polynomial features
10. Interaction features
11. Scaling
12. Missing-value indicators

The key principle is:

> **Create a feature because it represents a meaningful relationship — not simply because a new column can be created.**


In [18]:
# 7. Ratio features

# Add a tiny value to avoid division by zero.
eps = 1e-9

df["living_to_lot_ratio"] = df["sqft_living"] / (df["sqft_lot"] + eps)
df["bedrooms_per_1000_sqft"] = df["bedrooms"] / (df["sqft_living"] / 1000 + eps)

df[["living_to_lot_ratio", "bedrooms_per_1000_sqft"]].head()
# df.head()


,living_to_lot_ratio,bedrooms_per_1000_sqft
0,0.436594,0.497925
1,0.268989,0.498008
2,0.315229,0.606673
3,0.262340,0.543478
4,0.333542,0.625000


# Part D — Log, Square-Root and Power Transformations

Many real-world variables are **right-skewed**.

Examples:

- Income
- Sales
- House area
- Transaction amounts
- Population

A log transformation can compress very large values and make the distribution easier for some models to learn.

For zero-or-positive values:

`log1p(x) = log(1 + x)`

`log1p` is safer than `log(x)` when zeros exist.


In [19]:
# 10. Log transformations

positive_cols = [
    "sqft_living",
    "sqft_lot",
    "sqft_above",
    "sqft_basement",
    "sqft_living15",
    "sqft_lot15"
]

for col in positive_cols:
    df[col + "_log"] = np.log1p(df[col])

# Square-root transformation
df["sqft_living_sqrt"] = np.sqrt(df["sqft_living"].clip(lower=0))

display(df[[
    "sqft_living", "sqft_living_log", "sqft_living_sqrt"
]].head())


,sqft_living,sqft_living_log,sqft_living_sqrt
0,12050,9.396903,109.772492
1,10040,9.214432,100.199800
2,9890,9.199381,99.448479
3,9200,9.127067,95.916630
4,8000,8.987322,89.442719


## When should you use transformations?

- **Log:** strongly right-skewed positive variables
- **Square root:** moderate right skew, especially counts
- **Reciprocal:** useful in some relationships, but interpretability can decrease
- **Power transformations:** useful for reducing skewness
- **No transformation:** often perfectly fine for tree-based models

Feature engineering should be guided by the **model, distribution, domain meaning, and validation results**.


In [ ]:
# 11. Safe reciprocal and power examples

df["living_reciprocal"] = 1 / (df["sqft_living"] + eps)
df["living_squared"] = df["sqft_living"] ** 2
df["living_cubed"] = df["sqft_living"] ** 3

display(df[[
    "sqft_living", "living_reciprocal",
    "living_squared", "living_cubed"
]].head())


# Part E — Binning / Bucketing

Binning converts a continuous variable into meaningful groups.

Example:

`house_age → New / Medium / Old`

Binning can improve interpretability and can be useful when the relationship with the target is not linear.

However, binning also loses information, so do not use it automatically.


In [20]:
# 12. Binning numerical variables

df["age_group"] = pd.cut(
    df["house_age"],
    bins=[-1, 5, 15, 30, 60, np.inf],
    labels=["Very New", "New", "Middle", "Old", "Very Old"]
)

df["grade_group"] = pd.cut(
    df["grade"],
    bins=[0, 4, 7, 10, 13, np.inf],
    labels=["Low", "Average", "Good", "Very Good", "Excellent"]
)

display(df[[
    "house_age", "age_group", "grade", "grade_group"
]].head(10))


,house_age,age_group,grade,grade_group
0,104,Very Old,13,Very Good
1,74,Very Old,11,Very Good
2,13,New,13,Very Good
3,13,New,13,Very Good
4,6,New,12,Very Good
5,24,Middle,12,Very Good
6,15,New,12,Very Good
7,31,Old,12,Very Good
8,10,New,12,Very Good
9,13,New,12,Very Good


# Part F — Categorical Feature Engineering

Common categorical techniques:

- One-hot encoding
- Ordinal encoding
- Frequency/count encoding
- Rare-category grouping
- Binary encoding for two-category variables

### Important distinction

If categories have a real order:

`Low < Medium < High`

ordinal encoding may make sense.

If there is no natural order:

`Karachi, Lahore, Islamabad`

do **not** assign arbitrary numbers such as 1, 2, 3 and expect the model to understand them as equal-spaced numerical values.


In [21]:
# 13. Frequency encoding

# Frequency encoding replaces a category with its occurrence proportion.
zipcode_frequency = df["zipcode"].value_counts(normalize=True)

df["zipcode_frequency"] = df["zipcode"].map(zipcode_frequency)

display(df[["zipcode", "zipcode_frequency"]].head())


,zipcode,zipcode_frequency
0,98102,0.004858
1,98004,0.014667
2,98039,0.002313
3,98039,0.002313
4,98004,0.014667


In [22]:
# 14. One-hot encoding example

# In this dataset, age_group is a newly created categorical feature.
age_dummies = pd.get_dummies(
    df["age_group"],
    prefix="age",
    dtype=int
)

df = pd.concat([df, age_dummies], axis=1)

display(df[age_dummies.columns].head())


,age_Very New,age_New,age_Middle,age_Old,age_Very Old
0,0,0,0,0,1
1,0,0,0,0,1
2,0,1,0,0,0
3,0,1,0,0,0
4,0,1,0,0,0


## Ordinal encoding — simple example

For an ordered category such as:

`Low → Medium → High`

you can deliberately map the order:

`Low = 1, Medium = 2, High = 3`

Do this only when the order is meaningful.


In [23]:
# 15. Ordinal encoding example

example_order = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

example = pd.Series(["Low", "High", "Medium", "Low"])
encoded_example = example.map(example_order)

display(pd.DataFrame({
    "Original": example,
    "Encoded": encoded_example
}))


,Original,Encoded
0,Low,1
1,High,3
2,Medium,2
3,Low,1


# Part G — Missing Values as Features

Missingness itself can sometimes contain information.

For example:

- `income` is missing because a customer refused to provide it.
- `renovation_year` is missing because the house was never renovated.

You can create an indicator:

`is_missing = 1 if missing else 0`

Always combine this with a sensible missing-value treatment when the model requires complete numeric data.


# Part H — Latitude and Longitude Feature Engineering

Latitude and longitude are extremely valuable geographic variables.

Possible features include:

- Distance from a reference location
- Approximate distance from city center
- Latitude/longitude interactions
- Geographic zones
- Neighborhood density
- Clustering
- Distance to multiple important locations
- Haversine distance
- Spatial bins

### Important

Do not assume that simply treating latitude and longitude as ordinary numbers captures geographic distance perfectly. Longitude distance changes with latitude, and Earth's surface is curved.


In [25]:
# 17. Simple geographic features

# Interaction
df["lat_long_interaction"] = df["lat"] * df["long"]

# Geographic squares/bins
df["lat_bin"] = df["lat"].round(2)
df["long_bin"] = df["long"].round(2)

display(df[[
    "lat", "long", "lat_long_interaction",
    "lat_bin", "long_bin"
]].head())


,lat,long,lat_long_interaction,lat_bin,long_bin
0,47.6298,-122.323,-5826.220025,47.63,-122.32
1,47.6500,-122.214,-5823.497100,47.65,-122.21
2,47.6305,-122.240,-5822.352320,47.63,-122.24
3,47.6289,-122.233,-5821.823334,47.63,-122.23
4,47.6232,-122.220,-5820.507504,47.62,-122.22


In [27]:
# 18. Haversine distance function
# Calculates approximate distance between two latitude/longitude points.

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in kilometers

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


# Use the mean latitude/longitude as a simple reference point.
# In a real project, you could use an actual city center or business location.
reference_lat = df["lat"].mean()
reference_lon = df["long"].mean()

df["distance_from_reference_km"] = haversine_distance(
    df["lat"],
    df["long"],
    reference_lat,
    reference_lon
)

print("Reference point:", reference_lat, reference_lon)
display(df[[
    "lat", "long", "distance_from_reference_km"
]].head())


Reference point: 47.560052519317075 -122.21389640494147


,lat,long,distance_from_reference_km
0,47.6298,-122.323,11.273064
1,47.6500,-122.214,10.001707
2,47.6305,-122.240,8.074255
3,47.6289,-122.233,7.788366
4,47.6232,-122.220,7.036582


## Distance to multiple reference points

The same function can create features such as:

- Distance to airport
- Distance to school
- Distance to hospital
- Distance to office
- Distance to city center

This is especially useful in real-estate, logistics, retail, delivery and mobility datasets.


In [28]:
# 19. Multiple geographic distances — example
# Coordinates below are examples only. Replace them with real locations.

locations = {
    "center": (47.6062, -122.3321),
    "example_point_2": (47.4502, -122.3088)
}

for name, (ref_lat, ref_lon) in locations.items():
    df[f"distance_to_{name}_km"] = haversine_distance(
        df["lat"], df["long"], ref_lat, ref_lon
    )

display(df[[
    "lat", "long",
    "distance_to_center_km",
    "distance_to_example_point_2_km"
]].head())


,lat,long,distance_to_center_km,distance_to_example_point_2_km
0,47.6298,-122.323,2.711393,19.999035
1,47.6500,-122.214,10.101847,23.328175
2,47.6305,-122.240,7.413123,20.702933
3,47.6289,-122.233,7.845080,20.669150
4,47.6232,-122.220,8.612785,20.359012


# Part I — Aggregation Features

Aggregation features summarize groups.

Examples:

- Average house size within ZIP code
- Median income within city
- Number of transactions per customer
- Average purchase amount per customer

### Critical leakage warning

When building a predictive model, group statistics should generally be calculated using **training data only**, then mapped to validation/test data. Otherwise, information from the validation/test set can leak into the training process.


In [23]:
# 20. Demonstration of group aggregation
# This is useful for learning. In a real train/test workflow,
# calculate these statistics using training data only.

zip_median_living = df.groupby("zipcode")["sqft_living"].transform("median")
zip_mean_living = df.groupby("zipcode")["sqft_living"].transform("mean")
zip_house_count = df.groupby("zipcode")["zipcode"].transform("count")

df["zip_median_living"] = zip_median_living
df["zip_mean_living"] = zip_mean_living
df["zip_house_count"] = zip_house_count

df["living_vs_zip_median"] = (
    df["sqft_living"] / (df["zip_median_living"] + eps)
)

display(df[[
    "zipcode", "sqft_living",
    "zip_median_living", "zip_house_count",
    "living_vs_zip_median"
]].head())


,zipcode,sqft_living,zip_median_living,zip_house_count,living_vs_zip_median
0,98102,12050,1800.0,105,6.694444
1,98004,10040,2660.0,317,3.774436
2,98039,9890,3560.0,50,2.778090
3,98039,9200,3560.0,50,2.584270
4,98004,8000,2660.0,317,3.007519


# Part J — Polynomial and Interaction Features

Polynomial features allow a model to learn nonlinear relationships.

For example:

`x`

can become:

`x, x², x³`

Two variables can also produce interactions:

`x₁ × x₂`

Use these carefully because the number of columns can grow very quickly.


In [24]:
# 21. Simple manual polynomial features

df["sqft_living_squared"] = df["sqft_living"] ** 2
df["grade_squared"] = df["grade"] ** 2
df["age_squared"] = df["house_age"] ** 2

display(df[[
    "sqft_living", "sqft_living_squared",
    "grade", "grade_squared",
    "house_age", "age_squared"
]].head())


,sqft_living,sqft_living_squared,grade,grade_squared,house_age,age_squared
0,12050,145202500,13,169,104,10816
1,10040,100801600,11,121,74,5476
2,9890,97812100,13,169,13,169
3,9200,84640000,13,169,13,169
4,8000,64000000,12,144,6,36


# Part K — Scaling and Normalization

Scaling is technically a **preprocessing transformation**, but it is commonly included in the feature-engineering workflow.

### Standardization

Transforms a feature approximately to:

`mean = 0`

`standard deviation = 1`

### Min-Max scaling

Transforms values to a selected range, commonly:

`0 to 1`

### Robust scaling

Uses median and interquartile range and is often less sensitive to extreme values.

Tree-based models generally do not require scaling, while algorithms such as linear models, KNN, SVM and neural networks often benefit from appropriate scaling.


In [29]:
# 23. Scaling examples

scale_cols = ["sqft_living", "sqft_lot", "grade"]

sample = df[scale_cols].copy()

standard_scaler = StandardScaler()
sample_standard = standard_scaler.fit_transform(sample)

minmax_scaler = MinMaxScaler()
sample_minmax = minmax_scaler.fit_transform(sample)

robust_scaler = RobustScaler()
sample_robust = robust_scaler.fit_transform(sample)

display(pd.DataFrame(
    sample_standard,
    columns=[c + "_standard" for c in scale_cols]
).head())


,sqft_living_standard,sqft_lot_standard,grade_standard
0,10.855713,0.301622,4.545672
1,8.667171,0.536414,2.844170
2,8.503847,0.392738,4.545672
3,7.752557,0.481947,4.545672
4,6.445964,0.214344,3.694921


# Part L — Outlier-Aware Feature Engineering

Outliers can affect some transformations and models.

Useful approaches include:

- Log transformation
- Winsorization/capping
- Robust scaling
- Quantile-based bins
- Domain-based limits

Do not automatically delete every outlier. A luxury house with an unusually high price may be a valid observation.


In [27]:
# 24. Quantile-based capping example

def cap_by_quantile(series, lower=0.01, upper=0.99):
    low = series.quantile(lower)
    high = series.quantile(upper)
    return series.clip(lower=low, upper=high)

df["sqft_living_capped"] = cap_by_quantile(df["sqft_living"])

display(df[[
    "sqft_living", "sqft_living_capped"
]].describe())


,sqft_living,sqft_living_capped
count,21613.000000,21613.000000
mean,2079.899736,2070.894721
std,918.440897,872.755768
min,290.000000,720.000000
25%,1427.000000,1427.000000
50%,1910.000000,1910.000000
75%,2550.000000,2550.000000
max,13540.000000,4978.800000


# Part N — Leakage-Safe Train/Test Feature Engineering

This is one of the **most important ML concepts**.

Suppose you calculate:

`mean_income = mean(all data)`

before splitting the data.

The test set has influenced the transformation. That can cause **data leakage**.

A safer pattern is:

1. Split data
2. Fit transformations on training data
3. Apply those learned transformations to validation/test data

The same principle applies to:

- Imputation
- Scaling
- Frequency statistics
- Target encoding
- Group averages
- Feature selection
- PCA
- Many other learned transformations


In [29]:
# 26. Train/test split BEFORE fitting learned transformations

X = df.drop(columns=["price"], errors="ignore")
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)


Train: (17290, 61)
Test : (4323, 61)


In [30]:
# 27. Example: fit scaler on TRAINING data only

scale_columns = ["sqft_living", "sqft_lot", "grade"]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_columns] = scaler.fit_transform(
    X_train[scale_columns]
)

X_test_scaled[scale_columns] = scaler.transform(
    X_test[scale_columns]
)

display(X_train_scaled[scale_columns].head())


,sqft_living,sqft_lot,grade
6325,1.004060,-0.170179,1.139385
13473,-0.951984,-0.184153,-0.554798
17614,0.117896,-0.194852,-0.554798
16970,-0.314378,-0.194287,-0.554798
20868,-1.168121,-0.157006,-1.401890


# Part O — Feature Selection Basics

Feature engineering can create hundreds of columns. More features do **not automatically mean a better model**.

Consider:

- Domain relevance
- Missingness
- Variance
- Correlation
- Multicollinearity
- Model importance
- Redundancy
- Leakage
- Validation performance

A useful rule:

> **Keep features that add useful information, not simply features that exist.**


In [31]:
# 28. Low-variance / unique-value inspection

feature_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique_values": df.nunique(),
    "missing_pct": df.isna().mean() * 100
}).sort_values("unique_values")

display(feature_summary.head(20))


,dtype,missing,unique_values,missing_pct
waterfront,int64,0,2,0.0
sale_year,int32,0,2,0.0
is_weekend,int64,0,2,0.0
was_renovated,int64,0,2,0.0
sale_quarter,int32,0,4,0.0
condition,int64,0,5,0.0
view,int64,0,5,0.0
floors,float64,0,6,0.0
sale_weekday_name,object,0,7,0.0
dow_cos,float64,0,7,0.0


In [32]:
# 29. Correlation with the numeric target
# Correlation is NOT proof of causation and should not be the only
# feature-selection technique.

numeric_df = df.select_dtypes(include=np.number)

corr_with_price = (
    numeric_df.corr(numeric_only=True)["price"]
    .drop("price")
    .sort_values(key=abs, ascending=False)
)

display(corr_with_price.head(20))


sqft_living_squared             0.727439
sqft_living                     0.702035
grade_squared                   0.692402
sqft_living_capped              0.669640
grade                           0.667434
sqft_above                      0.605567
sqft_living15                   0.585379
living_vs_zip_median            0.547681
bathrooms                       0.525138
bedrooms_per_1000_sqft         -0.468367
zip_mean_living                 0.438264
living_vs_neighborhood          0.405392
zip_median_living               0.399983
view                            0.397293
sqft_basement                   0.323816
bedrooms                        0.308350
lat                             0.307003
lat_bin                         0.306996
living_vs_neighborhood_ratio    0.302192
distance_from_reference_km     -0.292970
Name: price, dtype: float64

# Part P — A Practical Feature-Engineering Checklist

Use this checklist when you receive a new dataset.

### 1. Understand the data
- What is the target?
- What does every column mean?
- Which columns are IDs?
- Which are numerical?
- Which are categorical?
- Which are dates?
- Which are text?
- Which are geographic?

### 2. Clean obvious problems
- Fix data types
- Handle missing values
- Check duplicates
- Check impossible values
- Investigate outliers

### 3. Engineer features
- Date parts
- Age/duration
- Ratios
- Differences
- Sums
- Interactions
- Log/power transformations
- Bins
- Categorical encoding
- Frequency/count encoding
- Missing indicators
- Geographic distances
- Group aggregations

### 4. Protect against leakage
- Split before fitting learned transformations
- Never use future information
- Be especially careful with target encoding and aggregations

### 5. Validate
Compare model performance **before vs after feature engineering**.

The feature is useful only if it contributes meaningful predictive information without leakage.


# Part Q — Final Feature Engineering Summary for KC House Data

This dataset gives us excellent examples of many real-world feature-engineering patterns:

| Raw feature | Engineered feature | Technique |
|---|---|---|
| `date` | `sale_year`, `sale_month` | Date extraction |
| `date` | `is_weekend` | Date logic |
| `date` | `month_sin`, `month_cos` | Cyclical encoding |
| `yr_built` | `house_age` | Domain feature |
| `yr_renovated` | `was_renovated` | Binary feature |
| `sqft_living` | `living_to_lot_ratio` | Ratio |
| `sqft_basement` | `basement_share` | Ratio |
| `sqft_living` | `sqft_living_log` | Log transformation |
| `house_age` | `age_group` | Binning |
| `zipcode` | `zipcode_frequency` | Frequency encoding |
| `lat`, `long` | `lat_long_interaction` | Interaction |
| `lat`, `long` | `distance_from_reference_km` | Geographic distance |
| `zipcode`, `sqft_living` | `zip_median_living` | Group aggregation |
| `sqft_living`, `grade` | polynomial terms | Polynomial features |
| numeric columns | standardized values | Scaling |

## The main lesson

**Raw data → meaningful features → machine-learning model**

Feature engineering is where **business/domain knowledge and machine learning meet**.


In [33]:
# 30. Final view of selected engineered features

selected_features = [
    "price",
    "house_age",
    "was_renovated",
    "living_to_lot_ratio",
    "bedrooms_per_1000_sqft",
    "bathrooms_per_bedroom",
    "basement_share",
    "living_vs_neighborhood_ratio",
    "sqft_living_log",
    "age_group",
    "zipcode_frequency",
    "distance_from_reference_km",
    "zip_median_living",
    "living_vs_zip_median",
    "grade_sqft_interaction"
]

available = [c for c in selected_features if c in df.columns]

display(df[available].head(10))
print("\nTotal columns after feature engineering:", df.shape[1])


,price,house_age,was_renovated,living_to_lot_ratio,bedrooms_per_1000_sqft,bathrooms_per_bedroom,basement_share,living_vs_neighborhood_ratio,distance_from_reference_km,zip_median_living,living_vs_zip_median
0,7700000,104,1,0.436594,0.497925,1.333333,0.288797,3.058376,11.273064,1800.0,6.694444
1,7062500,74,1,0.268989,0.498008,0.900000,0.235060,2.554707,10.001707,2660.0,3.774436
2,6885000,13,0,0.315229,0.606673,1.291667,0.104146,2.178414,8.074255,3560.0,2.778090
3,5570000,13,0,0.262340,0.543478,1.150000,0.326087,2.584270,7.788366,3560.0,2.584270
4,5350000,6,0,0.333542,0.625000,1.000000,0.160000,1.739130,7.036582,2660.0,3.007519
5,5300000,24,0,0.297636,0.811908,1.000000,0.323410,1.710648,0.447555,3020.0,2.447020
6,5110800,15,0,0.175978,0.624220,1.050000,0.252185,2.335277,12.972425,2175.0,3.682759
7,4668000,31,1,0.737680,0.518672,1.350000,0.500000,2.948012,0.447990,3020.0,3.192053
8,4500000,10,0,0.165942,0.753012,1.100000,0.043675,2.191419,21.617985,1660.0,4.000000
9,4489000,13,0,0.233674,0.622084,0.750000,0.000000,1.728495,6.765646,2660.0,2.417293



Total columns after feature engineering: 62


# Practice Exercises for Students

### Exercise 1 — Date
Create:
- Sale month
- Sale quarter
- Sale weekday
- Weekend flag

### Exercise 2 — House
Create:
- House age
- Was renovated?
- Years from construction to renovation

### Exercise 3 — Ratios
Create:
- Bathrooms per bedroom
- Living area / lot area
- Basement percentage

### Exercise 4 — Geography
Create:
- Latitude/longitude interaction
- Distance from the reference point

### Exercise 5 — Transformation
Create:
- Log of `sqft_living`
- Square root of `sqft_lot`
- Squared `grade`

### Exercise 6 — Categorical
Create:
- Age groups
- Frequency encoding of ZIP code

### Exercise 7 — Leakage
Split the data into train/test and explain why a scaler must be fitted only on training data.

### Challenge
Train a simple regression model using:
1. Raw features
2. Raw + engineered features

Compare the evaluation metrics and explain which engineered features appear useful.
